In [1]:
import pandas as pd
import sqlalchemy as sal

engine = sal.create_engine('mssql://ANKIT\SQLEXPRESS/master?driver=ODBC+DRIVER+17+FOR+SQL+SERVER')
conn=engine.connect()

In [81]:
def extract():
    df_products = pd.read_csv('products.txt')
    df_products_db = pd.read_sql_query("select * from product_dim where end_date = '9999-12-31' " , conn)
    return df_products,df_products_db

def transform(df_products,df_products_db):
    df_merged = pd.merge(df_products , df_products_db , how='inner' , on = 'product_id')
    update_rows= df_merged['product_key']
    keys = update_rows.to_list()
    product_keys= ','.join([str(key) for key in keys])
    return product_keys
    
def inserts(df_products):
    df_products['start_date'] = pd.to_datetime('now').strftime('%Y-%m-%d')
    df_products['end_date'] = '9999-12-31'
    df_products.to_sql('product_dim',con=conn , index=False , if_exists = 'append')
    conn.commit()
    
def updates(product_keys):
    query = sal.text("update product_dim set end_date =  cast(getdate()-1 as date) where product_key in (" + product_keys + ")")
    p = conn.execute(query)
    conn.commit()    

In [82]:
df_products,df_products_db = extract()


In [84]:
update_rows = transform(df_products,df_products_db)

In [98]:
if product_keys != '':
    updates(product_keys)

In [100]:
inserts(df_products)

In [88]:
update_rows

Series([], Name: product_key, dtype: int64)

In [83]:
df_products_db

,product_key,product_id,product_name,price,start_date,end_date
0,3,100,iPhone 13,60000,2023-06-29,9999-12-31
1,4,200,HP Laptop Pro,80000,2023-06-29,9999-12-31
2,5,300,iPhone 13 PRO,70000,2023-06-29,9999-12-31
3,7,400,iPhone 13 PRO PRO,90000,2023-06-29,9999-12-31


In [90]:
df_products

,product_id,product_name,price
0,400,iPhone 13 PRO PRO,95000


In [91]:
pd.merge(df_products , df_products_db , how='inner' , on = 'product_id')

,product_id,product_name_x,price_x,product_key,product_name_y,price_y,start_date,end_date
0,400,iPhone 13 PRO PRO,95000,7,iPhone 13 PRO PRO,90000,2023-06-29,9999-12-31


In [97]:
product_keys

'7'